# Spectral L-SML on Math Competition Benchmarks (Phase 13)

**Goal**: Run our spectral L-SML pipeline on the same models, datasets, and temperatures
as arXiv 2602.01288 (EDIS) and compare our results directly against their published numbers.

**EDIS reference numbers (arXiv 2602.01288, Section 5.3, Figure 5c):**
- EDIS score AUC = **0.804** (Qwen2.5-Math-1.5B, pooled 4 datasets × 3 temps)
- Mean entropy AUC = **0.673** (same setting)

**Our method**: L-SML — fully unsupervised, 1 forward pass, same access tier as EDIS.

| Method | Access | Labels | Role |
|--------|--------|--------|------|
| EDIS (paper) | Gray-box | None | reference |
| Mean entropy (paper) | Gray-box | None | reference |
| **L-SML (ours)** | Gray-box | None | computed — head-to-head |

In [1]:
# Cell 1 — Clone + pip install + imports
import os, sys, shutil

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

REPO_DIR = '/content/hallucination_detection'
if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)
if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b master https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull -q')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.system('pip install -q "transformers>=4.40" accelerate datasets bitsandbytes autoawq scipy scikit-learn')

from spectral_utils import (
    load_model, generate_full, free_memory,
    extract_all_features, FEAT_NAMES,
    load_cache, save_cache,
    boot_auc, sml_unsupervised,
)
from spectral_utils.data_loaders import (
    load_gsm8k, gsm8k_prompt, is_correct_gsm8k,
    load_math500, math_prompt, is_correct_math,
    load_amc23, amc23_prompt, is_correct_amc23,
    load_aime24, aime24_prompt, is_correct_aime24,
)
import numpy as np
import datasets  # freeze pyarrow before any gptqmodel install

print('spectral_utils imported OK')

spectral_utils imported OK


In [ ]:
# Cell 2 — Config
MODELS = [
    'Qwen/Qwen2.5-Math-1.5B-Instruct',   # primary — matches EDIS Section 5.2/5.3
    # 'Qwen/Qwen2.5-Math-7B-Instruct',   # deferred: run after 1.5B completes
    # 'Qwen/Qwen3-4B-Instruct',           # deferred: run after 1.5B completes
]
DATASETS  = ['gsm8k', 'math500', 'amc23', 'aime24']
TEMPS     = [0.2, 0.6, 1.0]
K         = 8      # candidates per problem — MUST match EDIS Section 5.3.
# EDIS Figure 5c AUC=0.804 pools all K=8 responses per problem as independent
# (score, label) pairs (26,356 total responses). To compare fairly, we do the
# same: each response is independently scored by L-SML and treated as one data
# point. K=1 would give valid AUC estimates but ~8x fewer data points and a
# different evaluation protocol than the paper. Cell 9 (Best-of-N selection)
# has been removed: our method is a 1-pass detection method, not a selection
# method, so that comparison does not apply.
MAX_NEW   = 1024
N_SAMPLES = {
    'gsm8k':   100,  # all done (T=0.2, T=0.6, T=1.0 cached)
    'math500':  50,  # cache has 50 problems done for T=0.2; T=0.6/T=1.0 need 50 each
    'amc23':    40,  # full dataset
    'aime24':   30,  # full dataset
}
CACHE_DIR = '/content/drive/MyDrive/hallucination_detection/cache/mathcomp_phase13'

PROMPT_FN  = {'gsm8k': gsm8k_prompt,  'math500': math_prompt,
              'amc23': amc23_prompt,   'aime24': aime24_prompt}
CORRECT_FN = {'gsm8k': is_correct_gsm8k, 'math500': is_correct_math,
               'amc23': is_correct_amc23,  'aime24': is_correct_aime24}

# EDIS paper reference (arXiv 2602.01288, Section 5.3, Figure 5c)
EDIS_REF_AUC    = 0.804
ENTROPY_REF_AUC = 0.673

# Offline consensus feature signs (higher = more likely correct)
# Derived from Step 110 reasoning-domain consensus (MATH/GSM8K cells)
FEATURE_SIGNS = {
    'epr': -1, 'trace_length': 1, 'spectral_entropy': -1,
    'low_band_power': -1, 'high_band_power': -1, 'hl_ratio': -1,
    'dominant_freq': -1, 'spectral_centroid': -1,
    'stft_max_high_power': -1, 'stft_spectral_entropy': -1,
    'rpdi': -1, 'sw_var_peak': -1,
    'pe_mean': -1, 'hurst_exponent': 1,
    'cusum_max': -1, 'cusum_shift_idx': 1,
}

print('Config OK')
total_per_model = sum(N_SAMPLES[d] for d in DATASETS) * len(TEMPS) * K
print(f'Total responses per model: {total_per_model:,}')
print(f'Models to run: {len(MODELS)} (1.5B primary; 7B and 4B deferred)')

In [3]:
# Cell 3 — Mount Google Drive + create cache dirs
from google.colab import drive
drive.mount('/content/drive')

for model_id in MODELS:
    slug = model_id.split('/')[-1].replace('.', '_')
    os.makedirs(f'{CACHE_DIR}/{slug}', exist_ok=True)

print(f'Cache root: {CACHE_DIR}')

Mounted at /content/drive
Cache root: /content/drive/MyDrive/hallucination_detection/cache/mathcomp_phase13


In [4]:
# Cell 4 — Pre-load datasets (once, before model loading)
DATASET_ROWS = {}
loaders = {
    'gsm8k':   lambda: load_gsm8k('test'),
    'math500': lambda: load_math500(n_samples=500),
    'amc23':   load_amc23,
    'aime24':  load_aime24,
}
for dname, fn in loaders.items():
    rows = fn()
    DATASET_ROWS[dname] = rows[:N_SAMPLES[dname]]
    print(f'  {dname}: {len(DATASET_ROWS[dname])} rows')

print('\nAll datasets loaded.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loaded 1319 GSM8K test problems.
  gsm8k: 100 rows
  lighteval/MATH_500 failed: Dataset 'lighteval/MATH_500' doesn't exist on the Hub or cannot be accessed.
Loaded 500 MATH problems from HuggingFaceH4/MATH-500.
  math500: 100 rows


README.md:   0%|          | 0.00/374 [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/40 [00:00<?, ? examples/s]

Loaded 40 AMC23 problems from math-ai/amc23.
  amc23: 40 rows


README.md:   0%|          | 0.00/1.78k [00:00<?, ?B/s]

aime_2024_problems.parquet:   0%|          | 0.00/37.2k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 30 AIME24 problems from Maxwell-Jia/AIME_2024.
  aime24: 30 rows

All datasets loaded.


In [ ]:
# Cell 5 — Inference loop
# For each (model, dataset, temperature): generate K=8 responses per problem.
# Each response stores the token-entropy trace and correctness label.
# Checkpoint every 25 problems.

import pickle, time

for MODEL_ID in MODELS:
    slug = MODEL_ID.split('/')[-1].replace('.', '_')
    print(f'\n=== Model: {MODEL_ID} ===')

    import re as _re, types as _types
    if 'gptq' in MODEL_ID.lower() or 'awq' in MODEL_ID.lower():
        _pcre = _types.ModuleType('pcre')
        for _fn in ('compile','match','search','findall','sub','split','fullmatch'):
            setattr(_pcre, _fn, getattr(_re, _fn))
        _pcre.error = _re.error
        for _flag in ('IGNORECASE','MULTILINE','DOTALL','VERBOSE','UNICODE','ASCII'):
            setattr(_pcre, _flag, getattr(_re, _flag))
        sys.modules['pcre'] = _pcre
        os.system('pip install -q --no-deps device-smi tokenicer defuser')
        os.system('pip install -q logbar')
        os.system('pip install -q --no-deps gptqmodel')

    mdl, tok = load_model(MODEL_ID, quantize_4bit=False)

    for dname in DATASETS:
        rows       = DATASET_ROWS[dname]
        prompt_fn  = PROMPT_FN[dname]
        correct_fn = CORRECT_FN[dname]

        for temp in TEMPS:
            cache_path = f'{CACHE_DIR}/{slug}/{dname}_T{temp:.1f}.pkl'
            key        = f'{slug}_{dname}_T{temp:.1f}'

            # Three-branch reload
            if key in globals() and globals()[key]:
                print(f'  [{dname} T={temp}] in memory ({len(globals()[key])} rows); skip')
                continue
            if os.path.exists(cache_path):
                with open(cache_path, 'rb') as f:
                    results = pickle.load(f)
                globals()[key] = results
                print(f'  [{dname} T={temp}] loaded from cache ({len(results)} rows)')
                if len(results) >= len(rows):
                    continue
                resume_idx = len(results)
            else:
                results    = []
                resume_idx = 0

            print(f'  [{dname} T={temp}] running from idx {resume_idx}...')
            t0 = time.time()

            for i, row in enumerate(rows[resume_idx:], start=resume_idx):
                prompt   = prompt_fn(row)
                traces   = []
                corrects = []
                for _ in range(K):
                    _r = generate_full(
                        mdl, tok, prompt,
                        temperature=temp, K=15, max_new_tokens=MAX_NEW
                    )
                    traces.append(_r['token_entropies'])
                    corrects.append(correct_fn(_r['full_text'], row))
                results.append({'idx': i, 'corrects': corrects, 'traces': traces})

                if (i + 1) % 25 == 0 or i + 1 == len(rows):
                    with open(cache_path, 'wb') as f:
                        pickle.dump(results, f)
                    print(f'    {i+1}/{len(rows)}  elapsed={time.time()-t0:.0f}s')

            globals()[key] = results
            acc = np.mean([any(r['corrects']) for r in results])
            print(f'  [{dname} T={temp}] done: acc={acc:.1%}')

    del mdl, tok
    free_memory()
    print(f'Unloaded {MODEL_ID}')


=== Model: Qwen/Qwen2.5-Math-1.5B-Instruct ===


config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-Math-1.5B-Instruct
  [gsm8k T=0.2] loaded from cache (100 rows)
  [gsm8k T=0.6] loaded from cache (100 rows)
  [gsm8k T=1.0] running from idx 0...
    25/100  elapsed=2976s
    50/100  elapsed=5694s
    75/100  elapsed=8163s
    100/100  elapsed=10848s
  [gsm8k T=1.0] done: acc=0.0%
  [math500 T=0.2] running from idx 0...
    25/100  elapsed=4918s
    50/100  elapsed=8927s


In [ ]:
# Cell 6 — Cache coverage check
import pickle

print('Cache coverage:\n')
for MODEL_ID in MODELS:
    slug = MODEL_ID.split('/')[-1].replace('.', '_')
    print(f'{MODEL_ID}:')
    for dname in DATASETS:
        for temp in TEMPS:
            path = f'{CACHE_DIR}/{slug}/{dname}_T{temp:.1f}.pkl'
            if os.path.exists(path):
                with open(path, 'rb') as f:
                    d = pickle.load(f)
                acc = np.mean([any(r['corrects']) for r in d]) if d else 0
                print(f'  {dname} T={temp:.1f}: {len(d)} problems, acc={acc:.1%}')
            else:
                print(f'  {dname} T={temp:.1f}: MISSING')

In [ ]:
# Cell 7 — Spectral feature extraction
# For each cached (model, dataset, temperature), extract the 16 spectral features
# from each entropy trace.
# Output: FEATURES[slug][dname][temp] = list of problem dicts, each with
#   corrects: list[bool]  (K labels)
#   feats_k:  list[dict|None]  (K feature dicts, None if trace too short)

import pickle

FEATURES = {}

for MODEL_ID in MODELS:
    slug = MODEL_ID.split('/')[-1].replace('.', '_')
    FEATURES[slug] = {}

    for dname in DATASETS:
        FEATURES[slug][dname] = {}

        for temp in TEMPS:
            path = f'{CACHE_DIR}/{slug}/{dname}_T{temp:.1f}.pkl'
            if not os.path.exists(path):
                print(f'  SKIP {slug}/{dname} T={temp} — cache missing')
                continue
            with open(path, 'rb') as f:
                raw = pickle.load(f)

            problems = []
            for r in raw:
                feats_k = [extract_all_features(tr) for tr in r['traces']]
                problems.append({
                    'idx':      r['idx'],
                    'corrects': r['corrects'],
                    'feats_k':  feats_k,
                })

            n_valid = sum(1 for p in problems for f in p['feats_k'] if f is not None)
            FEATURES[slug][dname][temp] = problems
            print(f'  {slug}/{dname} T={temp}: {len(problems)} problems, {n_valid} valid traces')

print('\nFeature extraction complete.')

In [ ]:
# Cell 8 — Per-response AUROC (Section 5.3 equivalent)
# Pool all K responses from all problems x datasets x temps for a given model.
# L-SML (fully unsupervised) is our method — head-to-head with EDIS 0.804.
# Mean entropy reproduction is a sanity check (should come out near 0.673).

from sklearn.metrics import roc_auc_score

def safe_auc(labels, scores):
    mask = ~np.isnan(scores)
    if mask.sum() < 10 or labels[mask].sum() == 0 or labels[mask].sum() == mask.sum():
        return np.nan
    return roc_auc_score(labels[mask], scores[mask])


def build_flat_arrays(slug, datasets=DATASETS, temps=TEMPS):
    """Flatten all (problem, candidate_k) pairs for a given model."""
    labels_l, ent_l, feats_rows = [], [], []

    for dname in datasets:
        for temp in temps:
            path = f'{CACHE_DIR}/{slug}/{dname}_T{temp:.1f}.pkl'
            if not os.path.exists(path):
                continue
            with open(path, 'rb') as f:
                raw = pickle.load(f)
            problems = FEATURES.get(slug, {}).get(dname, {}).get(temp)
            if problems is None:
                continue
            for r_raw, p in zip(raw, problems):
                for k_idx in range(len(p['corrects'])):
                    fl = p['feats_k'][k_idx]
                    tr = r_raw['traces'][k_idx]
                    labels_l.append(int(p['corrects'][k_idx]))
                    ent_l.append(-np.mean(tr) if tr else np.nan)
                    feats_rows.append(
                        [fl.get(fn, np.nan) if fl else np.nan for fn in FEAT_NAMES]
                    )

    labels = np.array(labels_l)
    feats  = np.array(feats_rows)
    return {
        'labels':     labels,
        'mean_ent':   np.array(ent_l),
        'feats':      feats,
        'feats_dict': {fn: feats[:, i] for i, fn in enumerate(FEAT_NAMES)},
    }


print('=== Per-response AUROC — L-SML vs EDIS paper ===')
print('Reference (arXiv 2602.01288, Fig. 5c):')
print(f'  EDIS score:   {EDIS_REF_AUC:.3f}')
print(f'  Mean entropy: {ENTROPY_REF_AUC:.3f}')
print()

for MODEL_ID in MODELS:
    slug = MODEL_ID.split('/')[-1].replace('.', '_')
    data = build_flat_arrays(slug)
    L    = data['labels']
    if len(L) == 0:
        print(f'\n{MODEL_ID}: no data — run Cell 5 first')
        continue

    print(f'\n--- {MODEL_ID} ---')
    print(f'N = {len(L):,}  |  accuracy = {L.mean():.1%}  '
          f'(EDIS paper: 36-49% for 1.5B on GSM8K)')

    ent_auc  = safe_auc(L, data['mean_ent'])
    lsml_auc = np.nan
    valid    = ~np.any(np.isnan(data['feats']), axis=1)
    if valid.sum() > 20 and L[valid].sum() > 5:
        fd = {fn: data['feats_dict'][fn][valid] for fn in FEAT_NAMES}
        try:
            lsml_scores, _ = sml_unsupervised(fd, FEAT_NAMES)
            lsml_auc = safe_auc(L[valid], lsml_scores)
        except Exception as e:
            print(f'  L-SML error: {e}')

    print(f'\n{"Method":<32} {"AUROC":<8} vs EDIS 0.804')
    print('-' * 55)
    print(f'{"EDIS (paper)":<32} {EDIS_REF_AUC:<8.3f} ---')
    print(f'{"Mean entropy (paper)":<32} {ENTROPY_REF_AUC:<8.3f} ---')
    print(f'{"Mean entropy (reproduced)":<32} {ent_auc:<8.3f} '
          f'{ent_auc-EDIS_REF_AUC:+.3f}  (sanity check ~0.673)')
    print(f'{"L-SML (ours)":<32} {lsml_auc:<8.3f} '
          f'{lsml_auc-EDIS_REF_AUC:+.3f}  <- head-to-head')

    print(f'\n  Individual features:')
    for fn in FEAT_NAMES:
        s   = data['feats_dict'][fn] * FEATURE_SIGNS.get(fn, 1)
        auc = safe_auc(L, s)
        bar = '▓' * int(max(0, auc - 0.5) * 40) if not np.isnan(auc) else ''
        print(f'  {fn:<24} {auc:.3f}  {bar}')

In [ ]:
# Cell 10 — Per-dataset L-SML AUROC breakdown
print('=== Per-dataset L-SML AUROC — Qwen2.5-Math-1.5B (pooled over T=0.2/0.6/1.0) ===')
print(f'{"Dataset":<12} {"N":<6} {"Acc%":<8} {"L-SML":<10} {"MeanEnt":<10} vs EDIS 0.804')
print('-' * 70)

slug_1b = 'Qwen2_5-Math-1_5B-Instruct'

for dname in DATASETS:
    all_labels, all_ent, all_feats = [], [], []

    for temp in TEMPS:
        path = f'{CACHE_DIR}/{slug_1b}/{dname}_T{temp:.1f}.pkl'
        if not os.path.exists(path):
            continue
        with open(path, 'rb') as f:
            raw = pickle.load(f)
        problems = FEATURES.get(slug_1b, {}).get(dname, {}).get(temp)
        if problems is None:
            continue
        for r_raw, p in zip(raw, problems):
            for k_idx in range(len(p['corrects'])):
                fl = p['feats_k'][k_idx]
                tr = r_raw['traces'][k_idx]
                all_labels.append(int(p['corrects'][k_idx]))
                all_ent.append(-np.mean(tr) if tr else np.nan)
                all_feats.append([fl.get(fn, np.nan) if fl else np.nan for fn in FEAT_NAMES])

    if not all_labels:
        print(f'{dname:<12} MISSING')
        continue

    L     = np.array(all_labels)
    Ent   = np.array(all_ent)
    F     = np.array(all_feats)
    valid = ~np.any(np.isnan(F), axis=1)

    ent_auc  = safe_auc(L, Ent)
    lsml_auc = np.nan
    if valid.sum() > 20 and L[valid].sum() > 5:
        fd = {fn: F[valid, i] for i, fn in enumerate(FEAT_NAMES)}
        try:
            scores, _ = sml_unsupervised(fd, FEAT_NAMES)
            lsml_auc  = safe_auc(L[valid], scores)
        except Exception:
            pass

    delta = f'{lsml_auc - EDIS_REF_AUC:+.3f}' if not np.isnan(lsml_auc) else 'N/A'
    print(f'{dname:<12} {len(L):<6} {L.mean():<8.1%} {lsml_auc:<10.3f} {ent_auc:<10.3f} {delta}')

In [ ]:
# Cell 11 — Final summary comparison table
print('=== Summary: L-SML vs EDIS paper ===')
print('Model: Qwen2.5-Math-1.5B  |  Datasets: GSM8K+MATH+AMC23+AIME24  |  T=0.2/0.6/1.0  |  K=8')
print()
print(f'{"Method":<35} {"Access":<12} {"Labels":<10} {"Pooled AUROC"}')
print('=' * 72)
print(f'{"EDIS (arXiv 2602.01288)":<35} {"Gray-box":<12} {"None":<10} 0.804')
print(f'{"Mean entropy (arXiv 2602.01288)":<35} {"Gray-box":<12} {"None":<10} 0.673')
print('-' * 72)

slug_1b = 'Qwen2_5-Math-1_5B-Instruct'
data    = build_flat_arrays(slug_1b)
L, F    = data['labels'], data['feats']
valid   = ~np.any(np.isnan(F), axis=1)

ent_auc = safe_auc(L, data['mean_ent'])
print(f'{"Mean entropy (reproduced)":<35} {"Gray-box":<12} {"None":<10} {ent_auc:.3f}  (sanity check)')

if valid.sum() > 50 and L[valid].sum() > 10:
    fd = {fn: F[valid, i] for i, fn in enumerate(FEAT_NAMES)}
    try:
        lsml_scores, meta = sml_unsupervised(fd, FEAT_NAMES)
        lsml_auc = safe_auc(L[valid], lsml_scores)
        delta    = f'{lsml_auc - EDIS_REF_AUC:+.3f}'
        print(f'{"L-SML (ours)":<35} {"Gray-box":<12} {"None":<10} {lsml_auc:.3f}  {delta} vs EDIS')
    except Exception as e:
        print(f'L-SML failed: {e}')
else:
    print('Insufficient data — run Cells 5-7 first.')

In [ ]:
# Cell 12 — Save results
import pickle

RESULTS_PATH = f'{CACHE_DIR}/spectral_lsml_vs_edis_results.pkl'
with open(RESULTS_PATH, 'wb') as f:
    pickle.dump({'FEATURES': FEATURES,
                 'config': {'MODELS': MODELS, 'DATASETS': DATASETS,
                            'TEMPS': TEMPS, 'K': K}}, f)
print(f'Saved to {RESULTS_PATH}')